# XGBoost Training

21 experiments: 7 feature sets and 3 split strategies.
1. 5-fold Optuna (TPE) hyperparameter search on the train set vs. 5-fold Random
2. Refit best params on full train set
3. Evaluate on test set (RMSE, MAE, R^2)

In [20]:
import sys, pathlib, os, json, time, warnings
import numpy as np
import pandas as pd
import joblib
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import uniform, randint

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# project root
ROOT = pathlib.Path(".").resolve().parent
sys.path.insert(0, str(ROOT))
from src.data_utils import load_experiment

MODELS_DIR = ROOT / "models" / "xgb"
RESULTS_DIR = ROOT / "results" / "xgb"

TARGET_IDX = 2  # co2_mol_kg_0.1bar
SEED = 42

In [21]:
# configuration
FEATURE_SETS = [
    "baseline",
    "geo_decorr", "geo_all",
    "rac_decorr", "rac_all",
    "combined_decorr", "combined_all",
]

SPLITS = ["random", "topology", "metal"]

N_TRIALS = 30  # Optuna trials
N_ITER    = 30  # RandomizedSearch iterations
CV_FOLDS  = 5

# RandomizedSearch search space
PARAM_DIST = {
    "n_estimators":     randint(100, 800),
    "max_depth":        randint(3, 11),
    "learning_rate":    uniform(0.01, 0.29),    # [0.01, 0.30]
    "subsample":        uniform(0.6, 0.4),       # [0.6, 1.0]
    "colsample_bytree": uniform(0.5, 0.5),       # [0.5, 1.0]
    "min_child_weight": randint(1, 10),
    "reg_alpha":        uniform(0, 1),
    "reg_lambda":       uniform(0.5, 1.5),
}

print(f"Experiments: {len(FEATURE_SETS)} feature sets × {len(SPLITS)} splits = {len(FEATURE_SETS)*len(SPLITS)}")
print(f"Per experiment: {CV_FOLDS} folds × {N_TRIALS} configs = {CV_FOLDS*N_TRIALS} fits")

Experiments: 7 feature sets × 3 splits = 21
Per experiment: 5 folds × 30 configs = 150 fits


Training loop

In [22]:
def _search_optuna(X, y):
    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

    def objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 800),
            "max_depth":        trial.suggest_int("max_depth", 3, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
            "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 9),
            "reg_alpha":        trial.suggest_float("reg_alpha", 0.0, 1.0),
            "reg_lambda":       trial.suggest_float("reg_lambda", 0.5, 2.0),
        }
        model = XGBRegressor(tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0, **params)
        scores = cross_val_score(model, X, y, cv=kf, scoring="neg_mean_squared_error")
        return float(np.sqrt(-scores.mean()))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    return study.best_params, study.best_value


def _search_random(X, y):
    xgb = XGBRegressor(tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0)
    search = RandomizedSearchCV(
        xgb,
        param_distributions=PARAM_DIST,
        n_iter=N_ITER,
        cv=CV_FOLDS,
        scoring="neg_mean_squared_error",
        random_state=SEED,
        n_jobs=-1,
        refit=False,
        verbose=0,
    )
    search.fit(X, y)
    return search.best_params_, float(np.sqrt(-search.best_score_))


def run_experiment(feature_set, split, method="optuna"):
    """Run one (feature_set × split × method) experiment.
    - CV hyperparameter search on the train set only.
    - Refit best model on full train set.
    - Evaluate on held-out test set (RMSE, MAE, R²).
    - If model already exists, load and re-evaluate instead of skipping.
    """
    tag = f"{feature_set}__{split}"
    model_dir   = MODELS_DIR / method
    results_dir = RESULTS_DIR / method
    model_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    model_path = model_dir / f"{tag}.joblib"

    X_train, X_test, y_train, y_test = load_experiment(
        split_strategy=split, feature_set=feature_set
    )
    y_tr = y_train[:, TARGET_IDX]
    y_te = y_test[:,  TARGET_IDX]

    if model_path.exists():
        print(f"  LOAD {tag} [{method}] (model exists, re-evaluating)", end="  ", flush=True)
        best = joblib.load(model_path)
        params_path = results_dir / f"{tag}_params.json"
        with open(params_path) as f:
            best_params = json.load(f)
        # Recompute CV RMSE with saved params so the metric is consistent
        kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
        cv_model = XGBRegressor(tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0, **best_params)
        cv_scores = cross_val_score(cv_model, X_train, y_tr, cv=kf, scoring="neg_mean_squared_error")
        best_cv_rmse = float(np.sqrt(-cv_scores.mean()))
    else:
        print(f"  {tag} [{method}]: X_train {X_train.shape} ...", end="  ", flush=True)

        if method == "optuna":
            best_params, best_cv_rmse = _search_optuna(X_train, y_tr)
        else:
            best_params, best_cv_rmse = _search_random(X_train, y_tr)

        best = XGBRegressor(tree_method="hist", random_state=SEED, n_jobs=-1, verbosity=0, **best_params)
        best.fit(X_train, y_tr)

        joblib.dump(best, model_path)
        with open(results_dir / f"{tag}_params.json", "w") as f:
            json.dump(
                {k: float(v) if isinstance(v, float) else int(v) for k, v in best_params.items()},
                f, indent=2,
            )

    y_pred    = best.predict(X_test)
    test_rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    test_mae  = float(mean_absolute_error(y_te, y_pred))
    test_r2   = float(r2_score(y_te, y_pred))

    result = {
        "feature_set":  feature_set,
        "split":        split,
        "method":       method,
        "n_features":   X_train.shape[1],
        "best_cv_RMSE": best_cv_rmse,
        "test_RMSE":    test_rmse,
        "test_MAE":     test_mae,
        "test_R2":      test_r2,
    }
    print(f"CV RMSE={best_cv_rmse:.4f}  |  test RMSE={test_rmse:.4f}  MAE={test_mae:.4f}  R²={test_r2:.4f}")
    return result


In [23]:
%%time

results = []
SUMMARY_PATH = RESULTS_DIR / "xgb_summary.csv"

for method in ["optuna", "random"]:
    print(f"\n{'='*50}")
    print(f"  Method: {method}")
    print(f"{'='*50}")
    for i, (fs, sp) in enumerate(
        [(fs, sp) for fs in FEATURE_SETS for sp in SPLITS], 1
    ):
        print(f"\n[{i}/21]", end=" ")
        t0 = time.time()
        r = run_experiment(fs, sp, method=method)
        if r is not None:
            r["time_min"] = round((time.time() - t0) / 60, 1)
            results.append(r)
            pd.DataFrame(results).to_csv(SUMMARY_PATH, index=False)

print(f"\nDone. {len(results)} new experiments saved.")



  Method: optuna

[1/21]   LOAD baseline__random [optuna] (model exists, re-evaluating)  CV RMSE=0.6617  |  test RMSE=0.6589  MAE=0.4621  R²=0.0305

[2/21]   LOAD baseline__topology [optuna] (model exists, re-evaluating)  CV RMSE=0.6614  |  test RMSE=0.9289  MAE=0.8360  R²=-1.0547

[3/21]   LOAD baseline__metal [optuna] (model exists, re-evaluating)  CV RMSE=0.6603  |  test RMSE=0.6670  MAE=0.4807  R²=0.0124

[4/21]   LOAD geo_decorr__random [optuna] (model exists, re-evaluating)  CV RMSE=0.3912  |  test RMSE=0.3867  MAE=0.2333  R²=0.6661

[5/21]   LOAD geo_decorr__topology [optuna] (model exists, re-evaluating)  CV RMSE=0.3892  |  test RMSE=0.3260  MAE=0.1838  R²=0.7469

[6/21]   LOAD geo_decorr__metal [optuna] (model exists, re-evaluating)  CV RMSE=0.4014  |  test RMSE=0.3769  MAE=0.2444  R²=0.6848

[7/21]   LOAD geo_all__random [optuna] (model exists, re-evaluating)  CV RMSE=0.3813  |  test RMSE=0.3736  MAE=0.2262  R²=0.6883

[8/21]   LOAD geo_all__topology [optuna] (model exists, 

Results summary

In [ ]:
df_res = pd.read_csv(RESULTS_DIR / "xgb_summary.csv")

# CV RMSE
print("CV RMSE (5-fold on train set)")
pivot_cv = df_res.pivot_table(index="feature_set", columns=["split", "method"], values="best_cv_RMSE")
pivot_cv = pivot_cv.reindex(FEATURE_SETS)
display(pivot_cv.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test RMSE 
print("\nTest RMSE")
pivot_test_rmse = df_res.pivot_table(index="feature_set", columns=["split", "method"], values="test_RMSE")
pivot_test_rmse = pivot_test_rmse.reindex(FEATURE_SETS)
display(pivot_test_rmse.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test R^2
print("\nTest R^2")
pivot_r2 = df_res.pivot_table(index="feature_set", columns=["split", "method"], values="test_R2")
pivot_r2 = pivot_r2.reindex(FEATURE_SETS)
display(pivot_r2.style.format("{:.4f}").background_gradient(cmap="RdYlGn"))

# Difference CV RMSE: Optuna - Random 
# if {"optuna", "random"}.issubset(df_res["method"].unique()):
#     delta = (
#         df_res.pivot_table(index=["feature_set", "split"], columns="method", values="test_RMSE")
#               .assign(delta=lambda d: d["optuna"] - d["random"])
#               .reset_index()
#     )
#     print("\nDifference in test RMSE = Optuna - Random:")
#     display(
#         delta.pivot(index="feature_set", columns="split", values="delta")
#              .reindex(FEATURE_SETS)
#              .style.format("{:+.4f}")
#              .background_gradient(cmap="RdYlGn_r")
#     )


CV RMSE (5-fold on train set)



Test RMSE



Test R^2



Difference in test RMSE = Optuna - Random:


split,metal,random,topology
feature_set,,,
baseline,-0.0047,-0.0000,+0.0587
geo_decorr,-0.0039,-0.0016,+0.0139
geo_all,+0.0017,-0.0054,+0.0082
rac_decorr,+0.0122,-0.0005,+0.0342
rac_all,+0.0116,-0.0006,-0.0048
combined_decorr,-0.0044,+0.0000,-0.0122
combined_all,-0.0060,-0.0025,+0.0236


In [26]:
df_res = pd.read_csv(RESULTS_DIR / "xgb_summary.csv")
df_opt = df_res[df_res["method"] == "optuna"]

# CV RMSE
print("CV RMSE (5-fold on train set)")
pivot_cv = df_opt.pivot_table(index="feature_set", columns="split", values="best_cv_RMSE")
pivot_cv = pivot_cv.reindex(FEATURE_SETS)
display(pivot_cv.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test RMSE
print("\nTest RMSE")
pivot_test_rmse = df_opt.pivot_table(index="feature_set", columns="split", values="test_RMSE")
pivot_test_rmse = pivot_test_rmse.reindex(FEATURE_SETS)
display(pivot_test_rmse.style.format("{:.4f}").background_gradient(cmap="RdYlGn_r"))

# Test R²
print("\nTest R^2")
pivot_r2 = df_opt.pivot_table(index="feature_set", columns="split", values="test_R2")
pivot_r2 = pivot_r2.reindex(FEATURE_SETS)
display(pivot_r2.style.format("{:.4f}").background_gradient(cmap="RdYlGn"))

CV RMSE (5-fold on train set)


split,metal,random,topology
feature_set,,,
baseline,0.6603,0.6617,0.6614
geo_decorr,0.4014,0.3912,0.3892
geo_all,0.3923,0.3813,0.3813
rac_decorr,0.6160,0.6141,0.6127
rac_all,0.6160,0.6137,0.6122
combined_decorr,0.3301,0.3205,0.3155
combined_all,0.3220,0.3111,0.3071



Test RMSE


split,metal,random,topology
feature_set,,,
baseline,0.6670,0.6589,0.9289
geo_decorr,0.3769,0.3867,0.3260
geo_all,0.3733,0.3736,0.3280
rac_decorr,0.6566,0.6093,0.6000
rac_all,0.6518,0.6088,0.5749
combined_decorr,0.3543,0.3191,0.3487
combined_all,0.3437,0.3064,0.3658



Test R^2


split,metal,random,topology
feature_set,,,
baseline,0.0124,0.0305,-1.0547
geo_decorr,0.6848,0.6661,0.7469
geo_all,0.6906,0.6883,0.7439
rac_decorr,0.0429,0.1710,0.1428
rac_all,0.0569,0.1723,0.2130
combined_decorr,0.7213,0.7726,0.7104
combined_all,0.7378,0.7904,0.6813
